In [ ]:
from datasets import load_dataset, DatasetDict
from unsloth import FastLanguageModel
import torch
from transformers import (
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

In [ ]:
DATASET_PATH = {
    "train": "train.path",
    "test": "test.path"
}
OUTPUT_DIR = "./qwen3-14b-rag-finetuned"
MODEL_NAME = "unsloth/Qwen3-14B-unsloth-bnb-4bit" 

In [ ]:
quantization_config = BitsAndBytesConfig(
    #загружать модель в четырехбитном формате
    load_in_4bit = True,

    #тип квантизации с битс энд байтса (похуй)
    bnb_4bit_quant_type = "nf4",  # Оптимальная квантизация для LLM

    #короче тоже похуй
    bnb_4bit_compute_dtype = torch.bfloat16,  # Совместимость с Ampere+ GPU
)

model, tokenizer = FastLanguageModel.from_pretrained(
    #название модели
    model_name = MODEL_NAME,

    #конфиг битс энд байтса (приводит модельку с произвольным размером фичей к фиксированному)
    # quantization_config=quantization_config, # для анслота не нужен

    #максимальная длина контекста модели (в токенах)
    max_seq_length = 8192,  # Удлиненный контекст для RAG

    #тип данных
    dtype = "bfloat16",

    #загрузить модель в 4 бита
    load_in_4bit = True,

    #использовать чекпоинты градиентов (чтобы не пересчитывать все с нуля а что-то за нас посчитают)
    use_gradient_checkpointing = True,  # Критично для экономии VRAM
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    #модель
    model,

    #ранг матриц исползуемых лорай для обучения (сильно растет используемая память и не сильно результат)
    r = 16,  # Rank для LoRA (оптимальный баланс качества/памяти)

    #вегда так писать так что похуй что это
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"], #всегда так писать

    #r * 2 всегде так что похуй
    lora_alpha = 32,

    #для борьбы с переобучением (убивается такое число весов за какоето время которое кирилл когда-то знал но забыл)
    lora_dropout = 0.05,

    #чуть улучшает скор но сильно увеличивает время и память (+const у весов)
    bias = "none",

    #чутьчуть оптимизирует вермя
    use_gradient_checkpointing = True,

    # тип рандомизации
    random_state = 42,
)

In [ ]:
dataset = load_dataset("csv", data_files = DATASET_PATH)
test_dataset = dataset["test"]

#функция файнтюна с рагом
def fine_rag(data):
    
    prompts = []
    answers = []
    
    for i in range(len(data['feature_1'])):
        prompt = f"""<|system|>
You are a helpful assistant. Predict the target value based on feature_1, ... feacture_n.
<|end|>

<|user|>
feature_1: {data["feature_1"][i]}
...
feature_n: {data["feature_n"][i]}
<|end|>

<|assistant|>
"""
    
        prompts.append(prompt)
        answers.append(data['target'][i])
        
    #токенизация промта
    tokenized_prompts = tokenizer(
        prompts,
        padding = True,
        max_length = 8192,
    )

    #токенизация ответа
    tokenized_answers = tokenizer(
        answers,
        padding = True,
        max_length = 4096,
    )

    input_ids = []
    labels = []
    attention_masks = []
    
    for i in range(len(prompts)):
        
        prompt_ids = tokenized_prompts.input_ids[i]
        answer_ids = tokenized_answers.input_ids[i]
        
        # Собираем полную последовательность: промпт + ответ
        full_ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]

        #Делаем маску внимания
        prompt_mask = tokenized_prompts.attention_mask[i]
        answer_mask = tokenized_answers.attention_mask[i]
        
        # Создаем маску внимания (квен сказал поставить все еденичкой чтобы не конфликтовать с датаколлатором)
        attention_mask = prompt_mask + answer_mask + [1]
        attention_masks.append(attention_mask)
        
        # Метки для обучения: -100 (игнорирование) для промпта, ответ для ответа
        label_ids = [-100] * len(prompt_ids) + answer_ids + [tokenizer.eos_token_id]
        
        input_ids.append(full_ids)
        labels.append(label_ids)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels,
    }


tokenized_dataset = dataset["train"].map(
    
    #функция
    fine_rag,

    # делить на батчи
    batched = True,

    #убрать колонки
    remove_columns = dataset["train"].column_names
)

In [ ]:
training_args = TrainingArguments(
    #директория сохранения
    output_dir = OUTPUT_DIR,
    
    #размер батча на трейне
    per_device_train_batch_size = 1,
    
    #размер батча на валидации
    per_device_eval_batch_size = 1,

    #накапливает память
    gradient_accumulation_steps = 8,

    #скорость обучения (не надо менять не ебу на самом деле че это)
    learning_rate = 2e-5,

    #число эпох
    num_train_epochs = 3,
 
    #как часто считает метрики
    logging_steps = 10,

    #как часто валидация
    eval_steps = 500,

    #как часто сохранение
    save_steps = 1000,

    #за сколько шагов дойдет до лернинг рейта
    warmup_steps = 100,

    #по какому временному параметру валидация
    eval_strategy = "steps",

    #по какому временному параметру сохранение
    save_strategy = "steps",

    #сохранить лучшую модель (если False то загрузит поседнюю)
    load_best_model_at_end = True,
 
    #по какой метрики вычисляется лучшая модель
    metric_for_best_model = "eval_loss",

    #больше - лучше соответственно с метрикой
    greater_is_better = False,

    #тип данных - float (16 после запятой)
    fp16 = True,

    #если True быстрее но бьет по VRAM, если False наоборот
    dataloader_pin_memory = False,

    #отчет о работе ллм
    report_to = None,

    #поиск неиспользуемых параметров
    ddp_find_unused_parameters = False,
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(

    #модель токенизации
    tokenizer,

    #выравнивает длину батча до ближайшего кратного
    pad_to_multiple_of = 8,

    #использовать паддинг
    padding = True,

    #определяет формат тензоров
    return_tensors = "pt",
)

In [ ]:
train_test_split = tokenized_dataset["train"].train_test_split(

    #размер авыборки
    test_size = 0.1,

    #способ перемешивания
    seed = 42,

    #надо ли перемешивать
    shuffle = True
)

#трейн выборка
train_dataset = train_test_split["train"]

#валидационная выборка
val_dataset = train_test_split["test"]

In [ ]:
trainer = Trainer(
    #модель
    model = model,

    #аргументы
    args = training_args,

    #тренировочный датасет
    train_dataset = train_dataset,

    #валидационный датасет
    eval_dataset = val_dataset,

    #как читать датасет
    data_collator = data_collator,

    #токенайзер
    tokenizer = tokenizer,
)

In [ ]:
print("Начало тонкой настройки...")
trainer.train()

# Сохраняем модель
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Модель успешно настроена и сохранена в {OUTPUT_DIR}")

In [ ]:
print("\nНачало генерации предсказаний для тестовой выборки...")

def preprocess_test(data):
    prompts = []
    answers = []
    
    for i in range(len(data['feature_1'])):
        prompt = f"""<|system|>
You are a helpful assistant. Predict the target value based on feature_1, ... feacture_n.
<|end|>

<|user|>
feature_1: {data["feature_1"][i]}
...
feature_n: {data["feature_n"][i]}
<|end|>

<|assistant|>
"""
    
        prompts.append(prompt)
    
    # Токенизация без меток
    inputs = tokenizer(
        prompts,
        padding=True,
        max_length=8192,
        return_tensors="pt",
        add_special_tokens=True
    )
    inputs["ids"] = ids
    return inputs

# Проверка наличия необходимых колонок
if "id" not in test_dataset.column_names:
    print("Предупреждение: В тестовом датасете нет колонки 'id'. Будут использованы индексы.")
    test_dataset = test_dataset.add_column("id", list(range(len(test_dataset))))

# Обработка тестовых данных
test_encodings = preprocess_test(test_dataset)

# Генерация предсказаний
model.eval()
predictions = []
batch_size = 1

for i in range(0, len(test_encodings["input_ids"]), batch_size):
    batch_ids = test_encodings["input_ids"][i:i+batch_size].to(model.device)
    batch_mask = test_encodings["attention_mask"][i:i+batch_size].to(model.device)
    
    outputs = model.generate(
        input_ids=batch_ids,
        attention_mask=batch_mask,
        max_new_tokens=64,
        do_sample=False,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )
    
    # Обработка вывода
    for j in range(batch_ids.shape[0]):
        # Вырезаем только сгенерированную часть
        prompt_length = batch_ids[j].shape[0]
        generated_ids = outputs[j, prompt_length:]
        
        # Удаляем специальные токены
        generated_text = tokenizer.decode(
            generated_ids, 
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        ).strip()
        
        # Очищаем от возможных артефактов
        generated_text = generated_text.split("<|")[0].strip()
        predictions.append(generated_text)

# Создание submission
submission = pd.DataFrame({
    "id": test_encodings["ids"],
    "target": predictions
})

# Сохранение результатов
submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)
print(f"Предсказания сохранены в {submission_path}")

# Показываем первые 5 результатов
print("\nПримеры предсказаний:")
print(submission.head())